## 7. Comparative Analysis & Recommendation

### Poređenje algoritama za `matheGesamt.csv`

| Kriterijum | IITA (Inductive Item Tree) | MIRT-VAE (Deep Learning) | NEAT (Neuroevolution) |
| :--- | :--- | :--- | :--- |
| **Priroda modela** | Deterministički (skupovni) | Probabilistički (kontinualni) | Evolutivni / Stohastički |
| **Ulazni podaci** | Binarni (0/1) | Binarni / Kategorijski | Binarni |
| **Otpornost na šum** | Srednja (zahteva parametre) | Visoka (inherentna regularizacija) | Visoka (populacioni pristup) |
| **Skalabilnost** | Niska (KVADRATNA složenost po itemima) | Visoka (linearna/batch po podacima) | Srednja (zavisi od populacije) |
| **Vizuelizacija (PHSG zahtev)** | **Odlična** (Hasse dijagrami) | Teška (Heatmape, Latent space scatter) | Dobra (Evoluirani graf) |
| **Spremnost za 2026.** | Mature / Legacy | Cutting Edge (Industry standard) | Experimental / Academic |

### Finalna Preporuka: Hibridni IITA + Vizuelizacija

S obzirom da je cilj projekta **"Konstruisati i vizuelizovati prostor znanja"** za pedagošku instituciju (PHSG), **interpretabilnost** je apsolutni prioritet. Crna kutija (VAE) teško objašnjava nastavniku *zašto* je neki put učenja preporučen.

**Preporučujem sledeći "pipeline" za Vaš naučni rad:**

1.  **Preprocessing:** Koristiti metode iz VAE (kao što je `Autoencoder` za denoising) samo za **čišćenje podataka** (popunjavanje 9999 i ispravljanje "nehatnih grešaka").
2.  **Core Algorithm:** Primeniti **Corrected IITA** ili **Structural KST** algoritam na očišćenim podacima da se dobije struktura grafa. Ovo će dati precizan Hasse dijagram koji se traži u specifikaciji.
3.  **Optimization (NEAT):** Ako IITA ne da zadovoljavajući fit, koristiti **Neuroevoluciju** samo za *finu optimizaciju* te strukture (dodavanje/brisanje grana radi maksimizacije likelihood-a).
4.  **Vizuelizacija:** Razviti web aplikaciju (npr. D3.js ili Python Streamlit) koja crta taj graf.

Ovaj pristup ("Denoising Autoencoder + IITA") kombinuje moć Deep Learninga (2026 tech) sa jasnoćom klasičnog KST-a (zahtev klijenta).

## 6. Algorithm Deep Dive: Neuroevolution (NEAT)

NEAT (NeuroEvolution of Augmenting Topologies) obično evoluira neuronske mreže. U kontekstu KST-a, možemo ga koristiti za evoluciju **strukture znanja** (grafa).

*   **Genom:** Predstavlja matricu povezanosti (adjacency matrix) između pojmova/zadataka.
*   **Fitness funkcija:** Meri koliko dobro predložena struktura objašnjava podatke ($\log L$). KST definiše da su *dozvoljena stanja* (knowledge states) zatvorena pod unijom i presekom.
*   **Evolucija:** Počinjemo od minimalne strukture (nema veza) i dodajemo veze (mutacije) koje povećavaju fitnes (objašnjavaju više studentskih odgovora uz minimalnu kompleksnost).

Ovo je inovativan pristup za 2026. jer izbegava lokalne optimume u koje upadaju pohlepni (greedy) algoritmi kao IITA.

**Predlog za implementaciju:** Koristiti `neat-python` biblioteku, gde bi "output" mreže bio predikcija odgovora, a topologija bi predstavljala zavisnosti. Međutim, direktna evolucija grafa (Q-matrix) je prirodnija.

## 5. Algorithm Deep Dive: MIRT-VAE (Deep Learning Approach)

MIRT-VAE koristi Variational Autoencoder arhitekturu.
*   **Encoder:** Prima vektor odgovora studenta $x$ (dimenzija $N_{items}$) i kompresuje ga u latentni vektor $z$ (dimenzija $K$, npr. 3-5 dimenzija znanja).
*   **Decoder:** Prima $z$ i rekonstruiše verovatnoće tačnog odgovora za svaki zadatak $\hat{x}$. Decoder funkcioniše kao IRT (Item Response Theory) model, gde težine mreže predstavljaju parametre zadataka (diskriminativnost i težinu).

Ovaj pristup je **generativan**. Možemo generisati nove validne obrasce odgovora i popunjavati praznine (missing data).

Za potrebe ovog notebooka, opisujemo arhitekturu koja bi se implementirala u PyTorch-u:

```python
# Pseudo-kod MIRT-VAE arhitekture
class MIRT_VAE(nn.Module):
    def __init__(self, n_items, latent_dim=5):
        super().__init__()
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(n_items, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim * 2) # Mu i LogVar
        )
        
        # Decoder (MIRT struktura: Logit = a*theta - b)
        # Ovde koristimo jednostavan Linear layer koji simulira IRT
        self.decoder = nn.Linear(latent_dim, n_items) 
        
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        h = self.encoder(x)
        mu, logvar = torch.chunk(h, 2, dim=-1)
        z = self.reparameterize(mu, logvar)
        
        # Rekonstrukcija (logits)
        logits = self.decoder(z)
        return logits, mu, logvar
```
Ovaj model automatski uči "težine" zadataka kroz težine u `decoder` sloju. Vizuelizacija bi zahtevala projektovanje $z$ prostora ili analizu težina decodera.

In [ ]:
def calculate_diff_values(data, items, threshold=0.05):
    """
    Jednostavna implementacija traženja potencijalnih preduslova.
    Tražimo parove (j, i) gde je j preduslov za i (j -> i).
    Očekujemo: ako student reši i, rešio je i j.
    Kontraprimer: student rešio i (1), a nije j (0).
    """
    n_students = len(data)
    implications = []
    
    # Sortiramo da smanjimo broj poređenja (samo lakši -> teži)
    # Ali brute-force je ok za manji broj itema.
    
    # Uzimamo subset itema za demo ako ih ima previše
    analyze_items = items[:50] if len(items) > 50 else items
    
    matrix = data[analyze_items].values
    n_items = len(analyze_items)
    
    print(f"Analiziram {n_items} zadataka za IITA simulaciju...")
    
    possible_relations = []
    
    for idx_j, item_j in enumerate(analyze_items):
        for idx_i, item_i in enumerate(analyze_items):
            if idx_i == idx_j:
                continue
            
            # Provera uslova težine: Preduslov (j) bi trebalo da bude lakši (češće rešavan) od (i)
            # P(j) >= P(i)
            p_j = np.mean(matrix[:, idx_j])
            p_i = np.mean(matrix[:, idx_i])
            
            if p_j < p_i:
                continue # j je teži od i, verovatno nije preduslov (osim ako su ekvivalenti)
            
            # Brojanje kontraprimera (i=1, j=0)
            # Koliko studenata zna teži (i), a ne zna lakši (j)
            counter_examples = np.sum((matrix[:, idx_i] == 1) & (matrix[:, idx_j] == 0))
            
            error_rate = counter_examples / n_students
            
            # Ako je error_rate mali, ovo je dobar kandidat za implikaciju j -> i
            if error_rate < threshold:
                possible_relations.append({
                    'prekursor': item_j,
                    'sledbenik': item_i,
                    'error_rate': error_rate,
                    'p_prekursor': p_j,
                    'p_sledbenik': p_i
                })
                
    return pd.DataFrame(possible_relations)

# Pokretanje analize na podacima
iita_results = calculate_diff_values(data_matrix, item_columns, threshold=0.02) # Strogi threshold od 2% greške

print(f"Pronađeno {len(iita_results)} potencijalnih relacija sa greškom < 2%.")
if not iita_results.empty:
    print(iita_results.sort_values('error_rate').head(10))
else:
    print("Nema relacija sa tako malom greškom. Povećati threshold.")

## 4. Algorithm Deep Dive: IITA (Data-Driven KST)

Ovde ćemo simulirati jednostavan IITA (Inductive Item Tree Analysis) proces. IITA traži relacije implikacije $i \implies j$ (ako znaš $i$, verovatno znaš i $j$, tj. $j$ je lakši/podskup od $i$). U KST terminologiji, često tražimo preduslove: da bi rešio $i$, moraš znati $j$ ($i \implies j$ ne važi, već $j$ prethodi $i$).

Konvencija u IITA: $j$ je preduslov za $i$ ako skoro niko ne rešava $i$ a da nije rešio $j$.
To znači: Slučaj ($i=1, j=0$) treba da bude redak. Težina $P(j) > P(i)$.

Brojaćemo kontrapimere za svaki par $(i, j)$ gde je $P(j) > P(i)$.
Kontraprimer $b_{ij}$: broj studenata koji su rešili $i$ (teži) a nisu $j$ (lakši).
Indeks difs ($d_{ij}$) meri koliko je ova implikacija narušena.

In [ ]:
# 1. Težina zadataka (Item Difficulty)
# U binarnim podacima, težina = procenat studenata koji su rešili zadatak (p-value).
# (Obrnuto: što je veći broj, zadatak je lakši)

item_difficulty = data_matrix.mean()

plt.figure(figsize=(12, 6))
sns.histplot(item_difficulty, bins=20, kde=True, color='skyblue')
plt.title('Distribucija težine zadataka (p-values)')
plt.xlabel('Procenat tačnih odgovora (Lakoća)')
plt.ylabel('Broj zadataka')
plt.axvline(0.5, color='red', linestyle='--')
plt.show()

print("Najteži zadaci (najmanji p-value):")
print(item_difficulty.sort_values().head(5))
print("\nNajlakši zadaci (najveći p-value):")
print(item_difficulty.sort_values(ascending=False).head(5))

# 2. Skorovi studenata
student_scores = data_matrix.sum(axis=1)

plt.figure(figsize=(12, 6))
sns.histplot(student_scores, bins=30, kde=True, color='green')
plt.title('Distribucija poena studenata')
plt.xlabel('Ukupan skor')
plt.ylabel('Broj studenata')
plt.show()

# 3. Correlation Matrix (Heatmap) - deo
# Ako su zadaci A i B jako korelisani, možda testiraju istu veštinu ili postoji zavisnost.
# Prikazaćemo samo deo matrice jer je verovatno velika.

subset_items = item_columns[:30] # Prvih 30 zadataka
corr_matrix = data_matrix[subset_items].corr()

plt.figure(figsize=(14, 12))
sns.heatmap(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1, square=True)
plt.title('Korelacija između prvih 30 zadataka')
plt.show()

## 3. Exploratory Data Analysis (EDA)

Analiziramo statistike zadataka i studenata kako bismo razumeli težinu i diskriminativnost testa.

## 2. Methodology Research: State of the Art (2026)

Na osnovu pregleda naučne literature relevantne za konstruisanje prostora znanja (Knowledge Space Theory - KST) i srodnih oblasti u 2026. godini, izdvajamo sledeće ključne pravce:

### 1. Klasični Deterministički Pristupi (IITA)
*   **Šta je to:** Inductive Item Tree Analysis (IITA) je familija algoritama za ekstrakciju *surmise* relacija (relacija preduslova) iz podataka.
*   **Status 2026:** I dalje se smatra "zlatnim standardom" za male i srednje skupove podataka gde je kritična **interpretabilnost**. Unapređene verzije (poput *Corrected IITA*) su robusnije na šum.
*   **Prednosti:** Daje eksplicitan grafon (DAG) koji pedagozi mogu odmah razumeti.
*   **Mane:** Teško skalira na hiljade zadataka; osetljiv na visoke stope "srećnog pogađanja" (lucky guess) i "nepažljivih grešaka" (careless errors) ako nisu eksplicitno modelovane.

### 2. Probabilistički Modeli & Latentni Prostori (MIRT-VAE)
*   **Šta je to:** Kombinacija Multidimensional Item Response Theory (MIRT) sa Variational Autoencoders (VAE).
*   **Status 2026:** Dominantan pristup u velikim adaptivnim sistemima (poput onih koje koriste Duolingo ili Khan Academy). Koristi duboke neuronske mreže da mapira studente u kontinualni latentni prostor, ali sa ograničenjima koja liče na KST strukture.
*   **Prednosti:** Izuzetno skalabilan, odlično barata nedostajućim podacima (sparse data), visoka prediktivna moć.
*   **Mane:** "Crna kutija". Teže je izvući eksplicitan graf preduslova (npr. "Za zadatak A moraš znati B") iz VAE latentnog prostora bez dodatnih tehnika (e.g., causal discovery).

### 3. Neuroevolucija (NEAT / HyperNEAT)
*   **Šta je to:** Korišćenje evolucionih algoritama za evoluiranje topologije mreže koja predstavlja prostor znanja.
*   **Status 2026:** Eksperimentalan, ali obećavajući pristup za otkrivanje nekonvencionalnih struktura. Umesto gradijentnog spusta (kao kod VAE), koristi se evolucija strukture grafa da se maksimizuje *likelihood* podataka.
*   **Prednosti:** Može otkriti proizvoljne topologije bez gausovskih pretpostavki koje nameće VAE.
*   **Mane:** Komputaciono veoma zahtevan (spor) za veće skupove podataka. Konvergencija nije garantovana.

**Zaključak pregleda:** Za projekat koji zahteva *vizuelizaciju* i jasnu strukturu za pedagoge (PHSG), interpretabilnost je ključna. KST/IITA nudi najbolju vizuelizaciju, dok VAE nudi bolje predviđanje. Hibridni pristup ili IITA su verovatno najpogodniji za ovaj specifičan dataset.

In [ ]:
# Čišćenje i Priprema Podataka
# Na osnovu inspekcije fajla:
# 0 = Netačno
# 1 = Tačno
# 9999 = Nedostajuća vrednost (verovatno nije radio zadatak)
# 666 = Verovatno nevalidan odgovor ili druga oznaka (tretiraćemo kao NaN za sada)

# Identifikujemo kolone sa zadacima. Prvih 5 kolona deluju kao metapodaci (standort, klasser, idr, BookletT1, BookletT2)
# Ostale kolone (s1m11a091, ...) su zadaci.

# Izdvajanje kolona zadataka
item_columns = [col for col in df.columns if col.startswith('s')]
metadata_columns = [col for col in df.columns if col not in item_columns]

print(f"Broj metapodataka: {len(metadata_columns)}")
print(f"Broj zadataka (items): {len(item_columns)}")

# Konverzija specijalnih vrednosti u NaN radi analize, ili u 0 (netačno) zavisno od pedagoške odluke.
# Za KST obično želimo binarne matrice (0/1). Ako student nije radio (9999), to je često "0" ili se ti podaci izbacuju.
# Ovde ćemo napraviti kopiju za analizu gde su 9999 i 666 označeni kao NaN da ne kvare statistiku,
# i drugu verziju gde su 0 (strogi kriterijum).

df_clean = df.copy()
df_clean[item_columns] = df_clean[item_columns].replace({9999: np.nan, 666: np.nan})

# Provera procenta nedostajućih vrednosti
missing_percentage = df_clean[item_columns].isnull().mean().mean()
print(f"Prosečan procenat nedostajućih podataka u matrici odgovora: {missing_percentage:.2%}")

# Kreiranje binarne matrice (fill NaN sa 0 - pretpostavka: nije znao/nije stigao = netačno)
# Ovo je standardna pretpostavka u mnogim testiranjima, ali treba biti pažljiv.
data_matrix = df[item_columns].replace({9999: 0, 666: 0})

# Osiguranje da su svi podaci 0 ili 1
# Ako ima drugih vrednosti (npr 2, 3), moramo znati.
unique_vals = np.unique(data_matrix.values)
print(f"Jedinstvene vrednosti u matrici odgovora nakon čišćenja: {unique_vals}")

data_matrix.head()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Konfiguracija za prikaz
%matplotlib inline
sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)

# Učitavanje podataka
try:
    # Pretpostavljamo da je separator tačka-zarez na osnovu pregleda fajla
    df = pd.read_csv('matheGesamt.csv', sep=';')
    print("Podaci uspešno učitani.")
    print(f"Dimenzije: {df.shape}")
except Exception as e:
    print(f"Greška pri učitavanju: {e}")

# Prikaz prvih nekoliko redova da vidimo strukturu
df.head()

# Analiza i Preporuka za Konstrukciju Prostora Znanja (KST) - Projekat 2026

Ovaj notebook sadrži detaljnu analizu podataka, pregled literature i preporuku algoritma za projekat "Konstruisanje prostora znanja za matematički domen nad realnim podacima".

Ciljevi:
1.  Učitavanje i čišćenje podataka (`matheGesamt.csv`).
2.  Pregled "State of the Art" metoda u 2026. godini (KST, Deep Learning, Neuroevolucija).
3.  Eksploratorna analiza podataka (EDA).
4.  Analiza tri potencijalna pristupa: IITA, MIRT-VAE, NEAT.
5.  Finalna preporuka.